## Simulating Sample Paths
### ACT460, Fall 2025

In this tutorial, we study different ways to simulate geometric Brownian motion.

In [8]:
# load Python packages
import numpy as np  # for numerical python libraries
import matplotlib.pyplot as plt  # for plotting

## Exact simulation of GBMs

In this example, we simulate exactly sample paths from a geometric Brownian motion (GBM) and inspect the effects of the different parameters. Recall that GBM has SDE
$$ dS_t = \mu S_t dt + \sigma S_t dW_t$$

In [9]:
def simulate_gbm_exact(S0, mu, sigma, T, Ndt, Nsims=10):
    """
    Simulate two-dimensional Geometric Brownian Motion with correlation rho.

    Parameters:
    S0 : float
        Initial value of the process.
    mu : float
        Drift coefficient of the process.
    sigma : float
        Volatility coefficient of the process.
    T : float
        Total time period to simulate.
    Ndt : float
        Number of time steps to take.

    Returns:
    prices : ndarray
        Simulated process over time.
    """
        
    # create time increment
    dt = T / Ndt
    
    # set the different times as a vector
    time = np.linspace(0, T, Ndt+1)
    
    # initialize the matrix for GBMs
    S = np.zeros((Ndt+1,Nsims))
    
    # initial value of the process
    S[0,:] = S0
    
    # recursively simulate the process ( S[idx,:] is (1 , Nsims) )
    for idx in range(Ndt):
        Z = np.random.randn(Nsims)
        # or Z = np.random.normal(0.0, 1.0, size=(1,Nsims))
        S[idx+1,:] = S[idx,:] * np.exp( (mu - (sigma**2)/2) * dt
                                      + sigma * np.sqrt(dt) * Z)
    
    return time, S

In [10]:


# Model parameters
S0 = 100
K = 95
r = 0.03
q = 0.02
sigma = 0.25
T = 0.5
m = 50

# Simulation settings
mu = r - q
Ndt = m
Nsims = 10000

# Random seed for reproducibility
np.random.seed(460)

time, S = simulate_gbm_exact(S0, mu, sigma, T, Ndt, Nsims)

S_monitor = S[1:, :]

log_S_monitor = np.log(S_monitor)
mean_log_S = np.mean(log_S_monitor, axis=0)
S_G = np.exp(mean_log_S)
payoff = np.maximum(K-S_G, 0)
discount = np.exp(-r * T)
P_G = np.mean(discount * payoff)
P_G_sd = np.std(discount * payoff, ddof=1) / np.sqrt(Nsims)
print(P_G)
P_G_sd = np.std(discount * payoff, ddof=1) / np.sqrt(Nsims)
print(P_G_sd)

1.9631263677358652
0.038581418327750236


In [11]:
from scipy.stats import norm

mu_G = np.log(100) + (r - q - sigma**2/2) * (T*(m+1))/(2*m)

v_G = sigma**2*T*(m+1)*(2*m+1)/(6*m**2)

d = (np.log(K) - mu_G)/np.sqrt(v_G)

A = K*norm.cdf(d)

B = np.exp(mu_G + v_G/2)*norm.cdf(d - np.sqrt(v_G))

P_G_a = np.exp(-r*T)*(A-B)

print(P_G_a)

1.9825017052583809


In [12]:
X = discount * payoff
S_A = np.mean(S_monitor, axis=0)
payoff_A = np.maximum(K-S_A, 0)
Y = discount * payoff_A
np.random.seed(461)
_, S_pilot = simulate_gbm_exact(
    S0, mu, sigma, T, Ndt, Nsims
)
S_pilot_monitor = S_pilot[1:, :]
S_A_pilot = np.mean(S_pilot_monitor, axis=0)
S_G_pilot = np.exp(np.mean(np.log(S_pilot_monitor), axis=0))
X_pilot = discount * np.maximum(K - S_G_pilot, 0)
Y_pilot = discount * np.maximum(K - S_A_pilot, 0)
b_hat = np.cov(Y_pilot, X_pilot, ddof=1)[0, 1] / np.var(X_pilot, ddof=1)
Z = Y - b_hat * (X - P_G_a)
P_MC = np.mean(Y)
P_CV = np.mean(Z)
MC_sd = np.std(Y, ddof=1) / np.sqrt(Nsims)
CV_sd = np.std(Z, ddof=1) / np.sqrt(Nsims)
CI_MC = (np.mean(Y) - 1.96 * MC_sd, np.mean(Y) + 1.96 * MC_sd)
CI_CV = (np.mean(Z) - 1.96 * CV_sd, np.mean(Z) + 1.96 * CV_sd)
Cor = np.corrcoef(X, Y)[0,1]
VRR = np.var(Y,ddof=1) / np.var(Z,ddof=1)
print("Optimal b:", b_hat)
print("MC Price:", P_MC)
print("CV Price:", P_CV)
print("MC Standard Error:", MC_sd)
print("CV Standard Error:", CV_sd)
print("MC 95% CI:", CI_MC)
print("CV 95% CI:", CI_CV)
print("Correlation:", Cor)
print("Variance Reduction Ratio:", VRR)

Optimal b: 0.9625845576782054
MC Price: 1.8738671800890472
CV Price: 1.8925175807880235
MC Standard Error: 0.03724002720905129
CV Standard Error: 0.001111427541285825
MC 95% CI: (np.float64(1.8008767267593067), np.float64(1.9468576334187877))
CV 95% CI: (np.float64(1.8903391828071032), np.float64(1.8946959787689437))
Correlation: 0.999557186251718
Variance Reduction Ratio: 1122.684354264667


In [13]:
import time

start_MC = time.perf_counter()

np.random.seed(461)
_, S_MC = simulate_gbm_exact(S0, mu, sigma, T, Ndt, Nsims)
S_MC_monitor = S_MC[1:, :]

S_A = np.mean(S_MC_monitor, axis=0)
payoff_A = np.maximum(K-S_A, 0)
Y = discount * payoff_A
P_MC = np.mean(Y)

t_MC = time.perf_counter() - start_MC

start_CV = time.perf_counter()

np.random.seed(461)

_, S_pilot = simulate_gbm_exact(
    S0, mu, sigma, T, Ndt, Nsims
)

S_pilot_monitor = S_pilot[1:, :]
S_A_pilot = np.mean(S_pilot_monitor, axis=0)
S_G_pilot = np.exp(np.mean(np.log(S_pilot_monitor), axis=0))

X_pilot = discount * np.maximum(K - S_G_pilot, 0)
Y_pilot = discount * np.maximum(K - S_A_pilot, 0)

b_hat = np.cov(Y_pilot, X_pilot, ddof=1)[0, 1] / np.var(X_pilot, ddof=1)

np.random.seed(462)

_, S_CV = simulate_gbm_exact(
    S0, mu, sigma, T, Ndt, Nsims
)

S_CV_monitor = S_CV[1:, :]

S_A_CV = np.mean(S_CV_monitor, axis=0)
S_G_CV = np.exp(np.mean(np.log(S_CV_monitor), axis=0))

X_CV = discount * np.maximum(K - S_G_CV, 0)
Y_CV = discount * np.maximum(K - S_A_CV, 0)

Z_CV = Y_CV - b_hat * (X_CV - P_G_a)
P_CV = np.mean(Z_CV)

t_CV = time.perf_counter() - start_CV

E_MC = np.var(Y, ddof=1) * t_MC / Nsims
E_CV = np.var(Z_CV, ddof=1) * t_CV / Nsims

print("MC Time:", t_MC)
print("CV Time:", t_CV)
print("MC Efficiency:", E_MC)
print("CV Efficiency:", E_CV)


MC Time: 0.013411100022494793
CV Time: 0.030580800026655197
MC Efficiency: 1.9483948696056524e-05
CV Efficiency: 3.8809967883621105e-08


In [14]:
sigmas = [0.15, 0.20, 0.25, 0.30, 0.35]
results = []
for sigma_current in sigmas:
    mu_G = np.log(100) + (r - q - sigma_current**2/2) * (T*(m+1))/(2*m)
    v_G = sigma_current**2*T*(m+1)*(2*m+1)/(6*m**2)
    d = (np.log(K) - mu_G)/np.sqrt(v_G)
    A = K*norm.cdf(d)
    B = np.exp(mu_G + v_G/2)*norm.cdf(d - np.sqrt(v_G))
    P_G_a_current = np.exp(-r*T)*(A-B)

    np.random.seed(461)

    _, S_pilot = simulate_gbm_exact(
        S0, mu, sigma_current, T, Ndt, Nsims
    )

    S_pilot_monitor = S_pilot[1:, :]
    S_A_pilot = np.mean(S_pilot_monitor, axis=0)
    S_G_pilot = np.exp(np.mean(np.log(S_pilot_monitor), axis=0))

    X_pilot = discount * np.maximum(K - S_G_pilot, 0)
    Y_pilot = discount * np.maximum(K - S_A_pilot, 0)

    b_hat = np.cov(Y_pilot, X_pilot, ddof=1)[0, 1] / np.var(X_pilot, ddof=1)

    np.random.seed(462)

    _, S_CV = simulate_gbm_exact(
        S0, mu, sigma_current, T, Ndt, Nsims
    )

    S_CV_monitor = S_CV[1:, :]

    S_A_CV = np.mean(S_CV_monitor, axis=0)
    S_G_CV = np.exp(np.mean(np.log(S_CV_monitor), axis=0))
    X_CV = discount * np.maximum(K - S_G_CV, 0)
    Y_CV = discount * np.maximum(K - S_A_CV, 0)
    Z_CV = Y_CV - b_hat * (X_CV - P_G_a_current)

    Corr = np.corrcoef(X_CV, Y_CV)[0,1]
    VRR = np.var(Y_CV,ddof=1) / np.var(Z_CV,ddof=1)
    P_CV = np.mean(Z_CV)

    results.append((sigma_current, Corr, VRR, P_CV))
print(results)



[(0.15, np.float64(0.999751131853036), np.float64(1998.2328462907565), np.float64(0.6307195834830326)), (0.2, np.float64(0.9996628767890441), np.float64(1477.590631382121), np.float64(1.2260002229290161)), (0.25, np.float64(0.9995517060413246), np.float64(1111.7251995020629), np.float64(1.8917993538757416)), (0.3, np.float64(0.9994244591258724), np.float64(866.0589091479327), np.float64(2.5958662402144155)), (0.35, np.float64(0.999275349151654), np.float64(688.149848030864), np.float64(3.322880958988366))]
